# 02 — Feature Engineering

**Purpose:** Build the complete feature table for all 48 teams and all training / prediction rows.
Every feature group is constructed and verified here. No new logic is introduced — all computation
delegates to `src/features.py`.

**Outputs**
- `outputs/team_features_freeze.parquet` — 48-row team feature table
- `outputs/training_rows.parquet` — 896-row training dataset
- `outputs/prediction_rows.parquet` — 112-row prediction dataset
- `outputs/feature_audit_report.csv` — per-feature null counts, ranges, validation flags

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.leakage_guard import (
    check_freeze_date, check_elo_snapshot,
    check_no_synthetic_data, check_training_rows_chronological,
    check_tactical_gap_preserved, check_form_window,
    FREEZE_DATE, TACTICAL_GAP_TEAMS,
)
from src.name_map import CANONICAL_48, WC_DEBUTANTS, canonicalize
from src.features import (
    load_ds2, load_ds4, load_ds6, load_ds8, load_ds10, load_ds11,
    load_ds16, load_ds17, load_ds18, load_ds19, load_ds1, load_ds1ext,
    build_elo_features, build_fifa_features, build_wc_historical_features,
    build_form_features, build_tournament_features, build_penalty_features,
    build_team_features, build_feature_table, build_training_rows,
)

DATA_ROOT = PROJECT_ROOT
ARC_BASE  = DATA_ROOT / "archive.zip"
ARC2      = DATA_ROOT / "archive (2).zip"      # DS8, DS10, DS11
ARC3      = DATA_ROOT / "archive (3).zip"      # DS1 (2026 WC results)
ARC4      = DATA_ROOT / "archive (4).zip"      # DS2 (Elo)
ARC6      = DATA_ROOT / "archive (6).zip"      # DS4, DS6
JUNE23    = DATA_ROOT / "june23_results.csv"   # DS1-ext: 4 June-23 matches
OUTPUTS   = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

print("Imports OK — loading raw datasets...")

Imports OK — loading raw datasets...


## Cell 2 — Load data and run leakage guards

In [6]:
ds2  = load_ds2(ARC4)
ds4  = load_ds4(ARC6)
ds6  = load_ds6(ARC6)
ds8  = load_ds8(ARC2)
ds10 = load_ds10(ARC2)
ds11 = load_ds11(ARC2)
ds16 = load_ds16(ARC_BASE)
ds17 = load_ds17(ARC_BASE)
ds1  = load_ds1(ARC3)               # 44 rows through June 22
ds1ext = load_ds1ext(JUNE23)        # 4 June-23 matches (score-only)

# Frozen 48-match result set
completed_48 = pd.concat([ds1, ds1ext], ignore_index=True)

# Freeze-filtered views for leakage-safe feature building
ds4_frozen = ds4[ds4["date"] <= pd.Timestamp("2026-06-23")].copy()
# snapshot_date is string-typed in DS2; use string comparison to avoid 0-row filter
ds2_frozen = ds2[
    (ds2["year"] == 2026) &
    (ds2["snapshot_date"] == "2026-05-27")
].copy()

# ── Leakage guards ────────────────────────────────────────────────────────
check_freeze_date(ds1,        date_column="date")
check_freeze_date(ds1ext,     date_column="date")
check_freeze_date(ds4_frozen, date_column="date")
check_elo_snapshot(ds2_frozen)

# form_dates: competitive matches in the form window only (2024-01-01 to 2026-06-10)
form_dates = ds4_frozen[
    (ds4_frozen["date"] >= "2024-01-01") &
    (ds4_frozen["date"] <= "2026-06-10")
]
check_form_window(form_dates)

print("✓ All leakage guards passed")
print(f"  DS2: {len(ds2_frozen)} Elo rows (snapshot) | DS4*: {len(ds4_frozen)} match rows | DS8: {len(ds8)} WC rows")
print(f"  DS11: {len(ds11)} FIFA ranking rows (2022-10-06)")
print(f"  completed_48: {len(completed_48)} rows  (DS1={len(ds1)} + DS1-ext={len(ds1ext)})")

✓ All leakage guards passed
  DS2: 48 Elo rows (snapshot) | DS4*: 49453 match rows | DS8: 964 WC rows
  DS11: 211 FIFA ranking rows (2022-10-06)
  completed_48: 48 rows  (DS1=44 + DS1-ext=4)


## Cell 3 — Elo features

In [7]:
elo_features = build_elo_features(ds2)

assert len(elo_features) == 48, f"Expected 48 Elo rows, got {len(elo_features)}"

# Design-spec spot checks
if "Spain" in elo_features.index:
    assert elo_features.loc["Spain", "elo_rating"] == 2165.0, "Spain Elo mismatch"
    print("✓ Spain elo_rating = 2165")

hosts = ["United States", "Canada", "Mexico"]
for h in hosts:
    if h in elo_features.index:
        assert elo_features.loc[h, "elo_is_host"] == 1, f"{h} should have is_host=1"
print(f"✓ Host flag verified for {[h for h in hosts if h in elo_features.index]}")

print("\nTop 10 teams by Elo rating:")
elo_features.sort_values("elo_rating", ascending=False).head(10)

✓ Spain elo_rating = 2165
✓ Host flag verified for ['United States', 'Canada', 'Mexico']

Top 10 teams by Elo rating:


,elo_rating,elo_rank,elo_rating_career_peak,elo_rating_career_avg,elo_is_host,confederation,elo_rank_in_wc_field
team_canonical,,,,,,,
Spain,2165,1,2189,1946,0,UEFA,1
Argentina,2113,2,2172,1987,0,CONMEBOL,2
France,2081,3,2135,1795,0,UEFA,3
England,2020,4,2213,1983,0,UEFA,4
Brazil,1984,5,2195,1998,0,CONMEBOL,5
Portugal,1984,5,2060,1797,0,UEFA,5
Colombia,1975,7,2069,1621,0,CONMEBOL,7
Netherlands,1961,8,2153,1848,0,UEFA,8
Ecuador,1933,9,1934,1525,0,CONMEBOL,9


## Cell 4 — FIFA features

In [8]:
fifa_features = build_fifa_features(ds10, ds11, elo_features)

assert len(fifa_features) == 48, f"Expected 48 FIFA rows, got {len(fifa_features)}"

print("FIFA rank vs Elo rank disagreement (top 10):")
if "elo_fifa_rank_disagreement" in fifa_features.columns:
    fifa_features[["elo_fifa_rank_disagreement"]].abs().sort_values(
        "elo_fifa_rank_disagreement", ascending=False
    ).head(10)

FIFA rank vs Elo rank disagreement (top 10):


## Cell 5 — WC historical features

In [9]:
wc_features = build_wc_historical_features(ds8, ds6)

assert len(wc_features) == 48, f"Expected 48 WC-hist rows, got {len(wc_features)}"

if "wc_debut_modern_flag" in wc_features.columns:
    debut_count = int(wc_features["wc_debut_modern_flag"].sum())
    print(f"Teams with wc_debut_modern_flag = 1: {debut_count}")
    print("Expected debutants:", sorted(WC_DEBUTANTS))

    # Verify known debutants have the flag set
    for team in WC_DEBUTANTS:
        if team in wc_features.index:
            assert wc_features.loc[team, "wc_debut_modern_flag"] == 1, (
                f"{team} should have debut flag"
            )
    print(f"✓ All {len(WC_DEBUTANTS)} known debutants verified")

wc_features.head()

Teams with wc_debut_modern_flag = 1: 7
Expected debutants: ['Cape Verde', 'Congo DR', 'Curaçao', 'Haiti', 'Iraq', 'Jordan', 'Uzbekistan']
✓ All 7 known debutants verified


,wc_tournaments_attended,wc_win_rate_modern,wc_win_rate_knockout_modern,wc_avg_gf_modern,wc_avg_ga_modern,wc_gd_per_game_modern,wc_clean_sheet_rate_modern,wc_best_result_encoded,wc_group_vs_knockout_uplift,wc_debut_modern_flag
team_canonical,,,,,,,,,,
Algeria,4,0.142857,0.000000,1.000000,1.285714,-0.285714,0.142857,2,-0.166667,0
Argentina,18,0.583333,0.400000,1.722222,1.000000,0.722222,0.416667,6,-0.314286,0
Australia,6,0.235294,0.000000,1.000000,1.882353,-0.882353,0.117647,2,-0.266667,0
Austria,7,0.000000,NaN,1.000000,1.333333,-0.333333,0.000000,5,NaN,0
Belgium,14,0.545455,0.571429,1.454545,0.954545,0.500000,0.363636,5,0.038095,0


## Cell 6 — Form features

In [10]:
form_features = build_form_features(ds4)

assert len(form_features) == 48, f"Expected 48 form rows, got {len(form_features)}"

print("Form features shape:", form_features.shape)

if "form_win_rate_last10" in form_features.columns:
    print("\nTop 10 by form win rate:")
    display(form_features[["form_win_rate_last10", "form_gd_last10"]]
            .sort_values("form_win_rate_last10", ascending=False).head(10))

Form features shape: (48, 6)

Top 10 by form win rate:


,form_win_rate_last10,form_gd_last10
team_canonical,,
England,1.0,3.0
Norway,1.0,4.0
Australia,0.8,1.6
Türkiye,0.8,1.2
Senegal,0.8,1.7
Morocco,0.8,1.6
Mexico,0.8,1.2
Croatia,0.8,2.2
Algeria,0.7,1.2


## Cell 7 — Tournament features (MD1 + MD2 in-tournament signal)

In [11]:
# build_tournament_features expects ds1 (44-row tactical DataFrame) and ds1ext (4-row scores-only)
tourn_features = build_tournament_features(ds1, ds1ext)

assert len(tourn_features) == 48, f"Expected 48 tournament rows, got {len(tourn_features)}"

# Design spec: exactly 4 teams have incomplete tactical data
if "has_full_tactical_md2" in tourn_features.columns:
    gap_teams = tourn_features[tourn_features["has_full_tactical_md2"] == 0].index.tolist()
    print(f"Teams with has_full_tactical_md2 = 0: {sorted(gap_teams)}")
    for t in TACTICAL_GAP_TEAMS:
        if t in tourn_features.index:
            assert tourn_features.loc[t, "has_full_tactical_md2"] == 0, (
                f"{t} should have incomplete tactical data"
            )
    print(f"✓ Tactical gap teams match design spec: {sorted(TACTICAL_GAP_TEAMS)}")

if "tourn_pts_md2" in tourn_features.columns:
    print("\nTop teams by tournament points:")
    display(tourn_features[["tourn_pts_md2", "tourn_gf_md2", "tourn_gd_md2"]]
            .sort_values("tourn_pts_md2", ascending=False).head(8))

Teams with has_full_tactical_md2 = 0: ['Colombia', 'Croatia', 'England', 'Portugal']
✓ Tactical gap teams match design spec: ['Colombia', 'Croatia', 'England', 'Portugal']

Top teams by tournament points:


,tourn_pts_md2,tourn_gf_md2,tourn_gd_md2
team_canonical,,,
Colombia,6,4,3
Argentina,6,5,5
United States,6,6,5
Mexico,6,3,3
Norway,6,7,4
Germany,6,9,7
France,6,6,5
Egypt,4,4,2


## Cell 8 — Penalty/shootout features

In [12]:
penalty_features = build_penalty_features(ds6, ds8)

assert len(penalty_features) == 48, f"Expected 48 penalty rows, got {len(penalty_features)}"

# Design spec spot checks (k=8 Bayesian shrinkage)
if "shootout_win_rate_alltime" in penalty_features.columns:
    for team, expected in [("Germany", 0.625), ("England", 0.40)]:
        if team in penalty_features.index:
            actual = float(penalty_features.loc[team, "shootout_win_rate_alltime"])
            print(f"{team}: shrunk rate = {actual:.4f}  (expected ≈ {expected})")

if "shootout_naive_flag" in penalty_features.columns:
    naive_teams = penalty_features[penalty_features["shootout_naive_flag"] == 1].index.tolist()
    print(f"\nTeams using naive shootout prior (no WC history): {sorted(naive_teams)}")

Germany: shrunk rate = 0.6250  (expected ≈ 0.625)
England: shrunk rate = 0.4000  (expected ≈ 0.4)

Teams using naive shootout prior (no WC history): ['Norway']


## Cell 9 — Assemble full team feature table

In [13]:
# build_team_features assembles all 6 feature groups into a 48-row team table.
# build_elo_features (called internally) filters ds2 to snapshot_date="2026-05-27",
# so the full ds2 must be passed here — not ds2_frozen.
# build_feature_table then expands it to a 96-row match-perspective table (48 matches × 2 teams).
team_features = build_team_features(
    ds2=ds2, ds10=ds10, ds11=ds11,
    ds8=ds8, ds6=ds6, ds4=ds4_frozen,
    ds1=ds1, ds1ext=ds1ext,
)

assert len(team_features) == 48, f"Expected 48 team rows, got {len(team_features)}"
print(f"Team feature table shape: {team_features.shape}")
print(f"  (expected: 48 rows, ~55 columns)")

# Leakage guard on assembled table
check_no_synthetic_data(team_features)
print("✓ check_no_synthetic_data passed")

# Build 96-row match-perspective prediction table (used for tournament simulation feature lookup)
ds18 = load_ds18(ARC_BASE)
ds19 = load_ds19(ARC_BASE)
prediction_rows = build_feature_table(
    team_features=team_features,
    ds16=ds16, ds17=ds17, ds18=ds18, ds19=ds19,
    ds1=ds1, ds1ext=ds1ext,
)
print(f"Prediction rows shape: {prediction_rows.shape}")
print(f"  (expected: ~96 rows, group-stage matches with outcomes)")

# Null counts
null_counts = team_features.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]
if not cols_with_nulls.empty:
    print(f"\nFeatures with nulls (≤12 expected for debutant WC features):")
    print(cols_with_nulls.to_string())
else:
    print("\n✓ No nulls in feature table")

team_features.head()

Team feature table shape: (48, 46)
  (expected: 48 rows, ~55 columns)
✓ check_no_synthetic_data passed
Prediction rows shape: (144, 65)
  (expected: ~96 rows, group-stage matches with outcomes)

Features with nulls (≤12 expected for debutant WC features):
wc_win_rate_modern              7
wc_win_rate_knockout_modern    21
wc_avg_gf_modern                7
wc_avg_ga_modern                7
wc_gd_per_game_modern           7
wc_clean_sheet_rate_modern      7
wc_group_vs_knockout_uplift    21
tourn_formation_changed         4
shootout_win_rate_wc_only      39


,elo_rating,elo_rank,elo_rating_career_peak,elo_rating_career_avg,elo_is_host,confederation,elo_rank_in_wc_field,fifa_points,fifa_points_delta,fifa_rank_delta,...,tourn_sot_conceded,tourn_shot_conversion_rate,tourn_yellow_cards_md2,tourn_formation_changed,has_full_tactical_md2,shootout_win_rate_alltime,shootout_appearances_total,shootout_win_rate_wc_only,shootout_first_shooter_advantage,shootout_naive_flag
team_canonical,,,,,,,,,,,,,,,,,,,,,
Spain,2165,1,2189,1946,0,UEFA,1,1873.013187,-3.382012,0,...,1.0,0.081633,1,0.0,1,0.500000,14,0.200000,0.357143,0
Argentina,2113,2,2172,1987,0,CONMEBOL,2,1876.118331,1.303496,2,...,0.5,0.227273,2,0.0,1,0.612903,23,0.857143,0.380952,0
France,2081,3,2135,1795,0,UEFA,3,1869.428449,-7.894282,-2,...,1.0,0.200000,0,0.0,1,0.473684,11,0.400000,0.545455,0
England,2020,4,2213,1983,0,UEFA,4,1827.048678,1.083196,0,...,5.0,0.181818,0,NaN,0,0.400000,12,0.250000,0.500000,0
Brazil,1984,5,2195,1998,0,CONMEBOL,5,1765.856297,4.695367,0,...,3.0,0.200000,3,1.0,1,0.541667,16,0.600000,0.562500,0


## Cell 10 — Build training rows

In [14]:
# build_training_rows requires: ds8, ds4, ds2 (full — needs historical year-end snapshots),
# ds6, ds10, ds11.
# Only matches where both teams are in CANONICAL_48 (2026 WC teams) are included.
training_rows = build_training_rows(ds8, ds4_frozen, ds2, ds6, ds10, ds11)

print(f"Training rows shape: {training_rows.shape}")
print(f"  ({len(training_rows)} rows = WC matches 1998–2022 where both teams in 2026 field)")

assert len(training_rows) > 0, "No training rows produced"
assert len(training_rows) % 2 == 0, f"Training rows must be even (home+away), got {len(training_rows)}"

if "outcome" in training_rows.columns:
    dist = training_rows["outcome"].value_counts().sort_index()
    print(f"\nOutcome distribution: {dist.to_dict()}")
    # WIN and LOSS counts must be equal (mirrored perspectives)
    assert dist.get(2, 0) == dist.get(0, 0), (
        f"WIN count ({dist.get(2,0)}) != LOSS count ({dist.get(0,0)}) — row mirroring broken"
    )
    print("✓ WIN == LOSS counts (mirroring correct)")

# L1 leakage guard: elo_year_used must equal match_year - 1
check_training_rows_chronological(training_rows)
print("\n✓ check_training_rows_chronological passed")

Training rows shape: (518, 37)
  (518 rows = WC matches 1998–2022 where both teams in 2026 field)

Outcome distribution: {0: 199, 1: 120, 2: 199}
✓ WIN == LOSS counts (mirroring correct)

✓ check_training_rows_chronological passed


## Cell 11 — Feature audit report

In [15]:
audit_rows = []
for col in team_features.columns:
    s = team_features[col]
    numeric_s = pd.to_numeric(s, errors="coerce")
    audit_rows.append({
        "feature":    col,
        "null_count": int(s.isnull().sum()),
        "dtype":      str(s.dtype),
        "min":        float(numeric_s.min()) if numeric_s.notna().any() else None,
        "max":        float(numeric_s.max()) if numeric_s.notna().any() else None,
        "mean":       float(numeric_s.mean()) if numeric_s.notna().any() else None,
    })

audit_df = pd.DataFrame(audit_rows)
print(f"Feature audit report: {len(audit_df)} features")

out_path = OUTPUTS / "feature_audit_report.csv"
audit_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

audit_df

Feature audit report: 46 features
Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/feature_audit_report.csv


,feature,null_count,dtype,min,max,mean
0,elo_rating,0,int64,1425.000000,2165.000000,1782.145833
1,elo_rank,0,int64,1.000000,95.000000,34.687500
2,elo_rating_career_peak,0,int64,1579.000000,2223.000000,1920.500000
3,elo_rating_career_avg,0,int64,1302.000000,1998.000000,1670.687500
4,elo_is_host,0,int64,0.000000,1.000000,0.062500
5,confederation,0,object,NaN,NaN,NaN
6,elo_rank_in_wc_field,0,int64,1.000000,48.000000,24.479167
7,fifa_points,0,float64,1275.581062,1876.118331,1580.765236
8,fifa_points_delta,0,float64,-7.894282,7.894282,0.960503
9,fifa_rank_delta,0,int64,-2.000000,2.000000,0.125000


## Cell 12 — Save outputs

In [16]:
paths = {
    "team_features_freeze.parquet": team_features,
    "training_rows.parquet":        training_rows,
}

for filename, df in paths.items():
    path = OUTPUTS / filename
    df.to_parquet(path, index=True)
    print(f"Saved → {path}  shape={df.shape}")

print("\n✓ All feature engineering outputs saved")

Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/team_features_freeze.parquet  shape=(48, 46)
Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/training_rows.parquet  shape=(518, 37)

✓ All feature engineering outputs saved
